# Practice 3 - Exercise 1: Sentiment Analysis với Hugging Face

## 1. Giới thiệu

Trong bài thực hành này, chúng ta sử dụng một mô hình phân tích cảm xúc đã được huấn luyện sẵn (pre-trained model) từ Hugging Face Hub để phân tích cảm xúc của văn bản.

Quy trình thực hiện:

**Văn bản → Tokenizer → Mô hình Pre-trained → Dự đoán cảm xúc**

In [6]:
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("PyTorch version      :", torch.__version__)
print("Transformers version :", transformers.__version__)
print("CUDA available       :", torch.cuda.is_available())

PyTorch version      : 2.13.0+cpu
Transformers version : 5.15.1
CUDA available       : False


## 2. Tải mô hình Pre-trained và Tokenizer

Sử dụng mô hình đã được huấn luyện sẵn từ Hugging Face Hub cho bài toán phân tích cảm xúc.

Tokenizer có nhiệm vụ chuyển văn bản thành dữ liệu số mà mô hình có thể xử lý. Mô hình pre-trained sau đó sử dụng dữ liệu này để dự đoán cảm xúc của văn bản.

In [7]:
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("Model     :", MODEL_NAME)
print("Tokenizer :", tokenizer.__class__.__name__)
print("Model type:", model.__class__.__name__)
print("Labels    :", model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model     : distilbert-base-uncased-finetuned-sst-2-english
Tokenizer : BertTokenizer
Model type: DistilBertForSequenceClassification
Labels    : {0: 'NEGATIVE', 1: 'POSITIVE'}


## 3. Tải mô hình phân tích cảm xúc đã được huấn luyện sẵn

Sử dụng một mô hình phân tích cảm xúc đã được huấn luyện sẵn từ Hugging Face Hub.

Mô hình được sử dụng là `distilbert-base-uncased-finetuned-sst-2-english`, có khả năng phân loại cảm xúc của văn bản thành hai nhãn: `NEGATIVE` và `POSITIVE`.

In [8]:
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("Pretrained model:", MODEL_NAME)
print("Model type      :", model.__class__.__name__)
print("Sentiment labels:", model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Pretrained model: distilbert-base-uncased-finetuned-sst-2-english
Model type      : DistilBertForSequenceClassification
Sentiment labels: {0: 'NEGATIVE', 1: 'POSITIVE'}


## 4. Tải Tokenizer

Tokenizer có nhiệm vụ chuyển văn bản đầu vào thành các token và giá trị số để mô hình có thể xử lý.

Tokenizer được tải từ cùng một checkpoint với mô hình để đảm bảo tính tương thích.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", tokenizer.__class__.__name__)

Tokenizer: BertTokenizer


## 5. Chuẩn bị câu văn mẫu

Chuẩn bị một câu tiếng Anh làm dữ liệu đầu vào để kiểm tra khả năng phân tích cảm xúc của mô hình.

Câu mẫu được chọn thể hiện cảm xúc tích cực rõ ràng để có thể dễ dàng kiểm tra kết quả dự đoán của mô hình.

In [10]:
sentence = "I really enjoyed this movie. It was amazing!"

print("Câu văn mẫu:")
print(sentence)

Câu văn mẫu:
I really enjoyed this movie. It was amazing!


## 6. Mã hóa văn bản bằng Tokenizer

Ở bước này, câu văn mẫu được đưa qua tokenizer để chuyển từ văn bản thành dữ liệu số mà mô hình có thể xử lý.

Kết quả mã hóa gồm:

- `input_ids`: các ID số đại diện cho các token trong câu.
- `attention_mask`: xác định những token mà mô hình cần chú ý khi xử lý.

In [11]:
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    padding=True,
    truncation=True
)

print("Kết quả Tokenization:")
print(inputs)

print("\nInput IDs:")
print(inputs["input_ids"])

print("\nAttention Mask:")
print(inputs["attention_mask"])

Kết quả Tokenization:
{'input_ids': tensor([[ 101, 1045, 2428, 5632, 2023, 3185, 1012, 2009, 2001, 6429,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Input IDs:
tensor([[ 101, 1045, 2428, 5632, 2023, 3185, 1012, 2009, 2001, 6429,  999,  102]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 7. Phân tích cảm xúc

Ở bước này, dữ liệu sau khi được tokenization sẽ được đưa vào mô hình pre-trained để dự đoán cảm xúc của câu văn.

Mô hình trả về `logits`, là các giá trị điểm số tương ứng với hai lớp cảm xúc:
- `NEGATIVE`
- `POSITIVE`

In [12]:
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

logits = outputs.logits

print("Logits:")
print(logits)

Logits:
tensor([[-4.3470,  4.6783]])


## 8. Hiển thị kết quả dự đoán

Các giá trị `logits` được chuyển thành xác suất bằng hàm Softmax.

Lớp có xác suất cao nhất được chọn làm kết quả dự đoán cuối cùng. Kết quả bao gồm:
- Nhãn cảm xúc (`POSITIVE` hoặc `NEGATIVE`).
- Độ tin cậy (`Confidence Score`) của mô hình.

In [13]:
# Chuyển logits thành xác suất
probabilities = torch.softmax(logits, dim=1)

# Lấy lớp có xác suất cao nhất
predicted_class_id = torch.argmax(probabilities, dim=1).item()

# Lấy label và confidence score
predicted_label = model.config.id2label[predicted_class_id]
confidence_score = probabilities[0][predicted_class_id].item()

print("Câu văn          :", sentence)
print("Cảm xúc dự đoán  :", predicted_label)
print(f"Độ tin cậy       : {confidence_score:.4f}")
print(f"Độ tin cậy (%)   : {confidence_score * 100:.2f}%")

Câu văn          : I really enjoyed this movie. It was amazing!
Cảm xúc dự đoán  : POSITIVE
Độ tin cậy       : 0.9999
Độ tin cậy (%)   : 99.99%


## 9. Giải thích kết quả

Mô hình dự đoán câu văn có cảm xúc **POSITIVE** với độ tin cậy **99.99%**.

Kết quả này phù hợp với nội dung của câu vì các từ như *enjoyed* và *amazing* thể hiện cảm xúc tích cực rõ ràng.

Quy trình phân tích được thực hiện như sau:

**Câu văn → Tokenizer → Input IDs / Attention Mask → Pre-trained Model → Logits → Xác suất → Sentiment**

Điều này cho thấy mô hình pre-trained có thể thực hiện phân tích cảm xúc trực tiếp mà không cần huấn luyện lại trong Exercise 1.

## 10. Kết luận

Trong Exercise 1, mô hình sentiment analysis đã được huấn luyện sẵn từ Hugging Face Hub được sử dụng để phân tích cảm xúc của một câu văn mẫu.

Quá trình thực hiện gồm:

**Câu văn → Tokenizer → Pre-trained Model → Dự đoán cảm xúc**

Kết quả cho thấy câu `"I really enjoyed this movie. It was amazing!"` được dự đoán là **POSITIVE** với độ tin cậy **99.99%**.

Qua bài thực hành, chúng ta đã thực hiện được các bước chính:
- Tải mô hình sentiment analysis đã được huấn luyện sẵn.
- Tải và sử dụng tokenizer tương ứng.
- Tokenize câu văn mẫu.
- Thực hiện sentiment analysis.
- Hiển thị và giải thích kết quả dự đoán.